In [ ]:
# movement スクレイピング用のモジュールを読み込みます
from pathlib import Path
import importlib

import polars as pl
import utils_scraping_seasearcher as sss



In [ ]:
# movement を取得したい vessel CSV を並べます
SOURCE_FILES = [
    str(Path("vessel") / "vessels_20260817_liquefiedgastanker.csv"),
    str(Path("vessel") / "vessels_20260817_oiltanker.csv"),
    str(Path("vessel") / "vessels_20260817_otherliquidstankers.csv"),
    str(Path("vessel") / "vessels_20260817_bulk.csv"),
    str(Path("vessel") / "vessels_20260817_chemicaltanker.csv"),
    str(Path("vessel") / "vessels_20260817_container.csv"),
    str(Path("vessel") / "vessels_202605_generalcargo.csv"),
    str(Path("vessel") / "vessels_202605_refrigeratedbulk.csv"),
    str(Path("vessel") / "vessels_20260817_roro.csv"),
    str(Path("vessel") / "vessels_202605_vehicle.csv"),
    str(Path("vessel") / "vessels_202605_cruise.csv"),
]

SOURCE_FILES


In [ ]:

# source CSV 群から Live の LLI を集めます
SOURCE_CONTEXT = sss.load_live_llinos_from_vessel_files(
    SOURCE_FILES,
    status_values=["Live"],
    unique=True,
    sort=True,
)

SOURCE_CONTEXT["source_summary"], SOURCE_CONTEXT["target_count"]


In [ ]:
# 実行条件をここで調整します
LOGIN_CONFIG = {
    "login_user": None,  # ???????????
    "login_password": None,  # ????????????
}

RUN_CONFIG = {
    "run_start": 0,
    "run_end": None,
    "reverse_targets": True,
    "max_workers": 1,
    "status_list": "all",
    "period": {"from": "2025-01-01", "to": "2026-07-31"},
    "local_time": False,
    "headless": False,
    "log_level": "DONE",
    "check_status": False,
    "log_steps": False,
    "skip_if_exists": True,
    "show_progress": True,
    "periodic_rest_enabled": True,
    "work_session_hours": 6.0,
    "work_session_random_minutes": 20.0,
    "rest_session_hours": 1.5,
    "rest_session_random_minutes": 15.0,
    "csv_download_timeout": 120,
    "csv_export_retry_attempts": 3,
    "next_page_ready_retry_attempts": 2,
}

resolved_status_list = sss.resolve_movement_status_list(RUN_CONFIG["status_list"])
status_slug = sss.build_movement_status_slug(RUN_CONFIG["status_list"])
period_label = sss.format_period_label(RUN_CONFIG["period"])
source_label = "csvset_4files"
OUTPUT_DIR = r"movement/20250101-20260526"

SCRAPING_CONFIG = {
    "max_hops": 10,
    "status_list": resolved_status_list,
    "period": RUN_CONFIG["period"],
    "local_time": RUN_CONFIG["local_time"],
    "out_dir": OUTPUT_DIR,
    "login_user": LOGIN_CONFIG["login_user"],
    "login_password": LOGIN_CONFIG["login_password"],
    "headless": RUN_CONFIG["headless"],
    "log_level": RUN_CONFIG["log_level"],
    "check_status": RUN_CONFIG["check_status"],
    "log_steps": RUN_CONFIG["log_steps"],
    "skip_if_exists": RUN_CONFIG["skip_if_exists"],
    "show_progress": RUN_CONFIG["show_progress"],
    "periodic_rest_enabled": RUN_CONFIG["periodic_rest_enabled"],
    "work_session_hours": RUN_CONFIG["work_session_hours"],
    "work_session_random_minutes": RUN_CONFIG["work_session_random_minutes"],
    "rest_session_hours": RUN_CONFIG["rest_session_hours"],
    "rest_session_random_minutes": RUN_CONFIG["rest_session_random_minutes"],
    "csv_download_timeout": RUN_CONFIG["csv_download_timeout"],
    "csv_export_retry_attempts": RUN_CONFIG["csv_export_retry_attempts"],
    "next_page_ready_retry_attempts": RUN_CONFIG["next_page_ready_retry_attempts"],
}

SCRAPING_CONFIG


In [ ]:
# 実際に流す LLI を確認します
# reverse_targets=True のときは全体を逆順にしてから run_start/run_end を適用します
ordered_targets = list(reversed(SOURCE_CONTEXT["targets"])) if RUN_CONFIG.get("reverse_targets", False) else SOURCE_CONTEXT["targets"]
targets_to_run = ordered_targets[RUN_CONFIG["run_start"]:RUN_CONFIG["run_end"]]
existing_preview = sss.preview_existing_movement_outputs(targets_to_run, SCRAPING_CONFIG)

target_summary = {
    "total_llino_in_source": SOURCE_CONTEXT["target_count"],
    "target_count_to_scrape": len(targets_to_run),
    "existing_output_count": existing_preview["existing_count"],
    "pending_after_skip_check": existing_preview["pending_count"],
    "run_start": RUN_CONFIG["run_start"],
    "run_end": RUN_CONFIG["run_end"],
    "reverse_targets": RUN_CONFIG.get("reverse_targets", False),
    "resolved_status_list": resolved_status_list,
    "period": RUN_CONFIG["period"],
    "out_dir": SCRAPING_CONFIG["out_dir"],
    "login_user": SCRAPING_CONFIG["login_user"] or "(default account)",
    "skip_if_exists": RUN_CONFIG["skip_if_exists"],
    "periodic_rest_enabled": RUN_CONFIG["periodic_rest_enabled"],
    "work_session_hours": RUN_CONFIG["work_session_hours"],
    "rest_session_hours": RUN_CONFIG["rest_session_hours"],
    "first_10_llino": targets_to_run[:10],
}

target_summary


In [ ]:
# movement をスクレイピングします
# このセルは全 targets の処理が終わるまで戻りません。まずは run_end を小さくして試すのがおすすめです。
results = sss.parallel_scraping(
    targets_to_run,
    max_workers=RUN_CONFIG["max_workers"],
    config=SCRAPING_CONFIG,
)

results[:5]
